# Acoustic finite-difference wavefield lab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_acoustic_fd_lab.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 5.1 Diffractions
2. 10.2 Capstone: Subtle Stratigraphic Trap
3. 9.6 Model QC
4. 12.3 Python Presets and Scaling Up

In [ ]:
# Ask Ogbon starter: Image a subtle or complex structure
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
TARGET = 'pinchout'
N = 300
F_MAX = 30.0
V_MIN, V_MAX = 1800.0, 4500.0

In [ ]:
# grid from the dispersion rule (8 points per wavelength), dt from CFL
H = V_MIN / (8.0 * F_MAX)
DT = 0.5 * H / (V_MAX * np.sqrt(2.0))
T = min(max(2.0 * N * H / V_MIN, 0.5), 4.0)
NT = int(T / DT)
SX, SZ = N // 2, 4
print('h = %.1f m, dt = %.2f ms, %d steps' % (H, DT*1000, NT))

v = np.full((N, N), V_MIN)
v[N//3:, :] = 0.5 * (V_MIN + V_MAX)
v[int(N*0.6):, :] = V_MAX
wedge_top = int(N*0.45)
for ix in range(N//2):                     # a thin wedge that pinches out
    thick = max(0, int((N//2 - ix) * 0.08))
    v[wedge_top:wedge_top+thick, ix] = V_MAX

c2 = (v * DT / H) ** 2
u0 = np.zeros((N, N)); u1 = np.zeros((N, N))
record = np.zeros((NT, N))

def src(it, f):
    a = (np.pi * f * (it * DT - 1.0 / f)) ** 2
    return (1 - 2*a) * np.exp(-a)

for it in range(NT):
    lap = (np.roll(u1, 1, 0) + np.roll(u1, -1, 0) +
           np.roll(u1, 1, 1) + np.roll(u1, -1, 1) - 4*u1)
    u2 = 2*u1 - u0 + c2 * lap
    u2[SZ, SX] += src(it, F_MAX * 0.6) * DT**2 * 1e6
    u2[0, :] = u2[-1, :] = u2[:, 0] = u2[:, -1] = 0
    record[it] = u2[2, :]                    # surface receivers
    u0, u1 = u1, u2

fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
m = np.abs(u1).max() * 0.3 + 1e-12
ax[0].imshow(u1, cmap='seismic', vmin=-m, vmax=m)
ax[0].set_title('wavefield snapshot')
r = np.abs(record).max() * 0.3 + 1e-12
ax[1].imshow(record, cmap='gray', vmin=-r, vmax=r, aspect='auto')
ax[1].set_title('surface record: diffractions live here')
plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)